# Python程序设计（项目实战） · 工单P5《产线数据仓库与ORM》 — 教师参考版
第7周 | 参考实现可直接运行验证 | 答案请勿下发学生

In [1]:
# --- 环境与数据准备(勿改, 运行即可) ---
import os, sys, subprocess
os.chdir(os.path.abspath("../"))          # 切到课程包根目录(data/所在处)
print("工作目录:", os.getcwd())


工作目录: /Users/joelgong/Desktop/现代制造 1/Python编程实战/教学实施资料/03-Python程序设计-项目实战


## 参考实现 `p55`（含依赖导入）

In [2]:
import sys
import numpy as np  # 通用科学计算


def p55():
    from sqlalchemy import create_engine, select, func
    from sqlalchemy.orm import DeclarativeBase, Session, relationship
    from sqlalchemy import Column, DateTime, Float, ForeignKey, Integer, String
    import datetime
    import random

    class Base(DeclarativeBase):
        pass

    class Dev(Base):
        __tablename__ = "devices"
        id = Column(Integer, primary_key=True)
        code = Column(String, unique=True)

    class Rec(Base):
        __tablename__ = "records"
        id = Column(Integer, primary_key=True)
        device_id = Column(Integer, ForeignKey("devices.id"))
        ts = Column(DateTime, default=datetime.datetime.utcnow)
        vib = Column(Float)

    e = create_engine("sqlite:///ref_warehouse.db")
    Base.metadata.drop_all(e)
    Base.metadata.create_all(e)
    random.seed(1)
    with Session(e) as s:
        s.add_all([Dev(code=f"DEV-{i:02d}") for i in range(1, 4)])
        s.add_all([Rec(device_id=random.randint(1, 3),
                       ts=datetime.datetime(2026, 4, 1) +
                       datetime.timedelta(minutes=i),
                       vib=random.uniform(1, 6)) for i in range(500)])
        s.commit()
        print("按设备平均振动:",
              s.execute(select(Dev.code, func.avg(Rec.vib))
                        .join(Rec, Rec.device_id == Dev.id)
                        .group_by(Dev.code)).all())
    with e.connect() as c:
        import time
        t0 = time.time()
        c.execute(select(Rec).where(Rec.device_id == 1).limit(200)).fetchall()
        print("查询耗时 %.2fms (加索引后对比见工单)" % ((time.time() - t0) * 1000))


## 学生版TODO → 教师答案对照
学生版starter中的TODO与上方参考实现（或下方双解）的对应关系：

| TODO | 题面（学生看到的指导） |
|------|----------------------|
| `1` | TODO-1: 补充 area(车间) 字段 |
| `2` | TODO-2: 联合索引 Index('idx_dev_ts', point_id, ts) |
| `3` | TODO-3: 种子数据: 3设备×各2测点 + 模拟记录500条(vib随机, 5%>4.5触发告警) |
| `4a` | TODO-4a: 各设备平均vib (JOIN+GROUP BY → func.avg) TODO-4b: |
|  | 告警Top3设备 TODO-4c: 按小时聚合记录数(func.date_trunc不可用→strftime) |

> 发放纪律：本对照表**勿随学生版下发**；课堂按层放提示（D层演示照做/C层只给思路行）。

## 多解对比（已实跑互验·三解）
| 解 | 路线 | 结论 |
|----|------|------|
| A | 裸SQL sqlite3 | DBA视角/复杂报表 |
| B | SQLAlchemy ORM | 类型安全/可迁移PostgreSQL（500条聚合与A一致） |
| C | pandas.read_sql | 要接着做可视化/透视 |

> 决策素材：教学主线B（工单口径），A给DBA路线，C给分析路线。

## 执行与验证

In [3]:
p55()

按设备平均振动: [('DEV-01', 3.5389503463999032), ('DEV-02', 3.607010482060574), ('DEV-03', 3.429850186954746)]
查询耗时 0.91ms (加索引后对比见工单)
